In [1]:
from pathlib import Path
import os
import sys
import time
import traceback

import torch
import whisper


In [2]:
def ensure_ffmpeg_on_path(ffmpeg_bin_dir: Path) -> None:
    if ffmpeg_bin_dir.exists() and ffmpeg_bin_dir.is_dir():
        path_str = os.environ.get("PATH", "")
        ffmpeg_str = str(ffmpeg_bin_dir)
        if ffmpeg_str not in path_str:
            os.environ["PATH"] = ffmpeg_str + os.pathsep + path_str

In [3]:
cwd = Path.cwd()

candidates = [
    cwd / "STT" / "whisper" / "test_data" / "type",
    cwd / "test_data" / "type",
]

type_dir = next((p for p in candidates if p.exists()), None)
if type_dir is None:
    raise FileNotFoundError(
        "Could not locate 'STT/whisper/test_data/type'. Please run this notebook from the project root or from 'STT/whisper'."
    )

result_dir = type_dir / "result"
result_dir.mkdir(parents=True, exist_ok=True)

In [4]:
ffmpeg_bin = (cwd / "STT" / "whisper" / "ffmpeg" / "bin")
if not ffmpeg_bin.exists():
    ffmpeg_bin = Path(__file__).resolve().parent / "ffmpeg" / "bin" if "__file__" in globals() else (Path.cwd() / "ffmpeg" / "bin")
ensure_ffmpeg_on_path(ffmpeg_bin)

In [5]:
model_size = "medium"  # change to "base", "medium", "large" if needed
device = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model(model_size, device=device)
use_fp16 = device == "cuda"
print("Device:", device)

c:\Users\chan\anaconda3\envs\poc-ai\lib\site-packages\whisper\__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_location=device)

Device: cuda


In [7]:
AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}
audio_files = sorted([p for p in type_dir.iterdir() if p.suffix.lower() in AUDIO_EXTS and p.is_file()])

if not audio_files:
    raise RuntimeError(f"No audio files found in: {type_dir}")

print(f"Found {len(audio_files)} audio files in {type_dir}")
print(f"Transcripts will be saved to: {result_dir}")

Found 5 audio files in c:\Users\chan\poc-story-field-ai\STT\whisper\test_data\type
Transcripts will be saved to: c:\Users\chan\poc-story-field-ai\STT\whisper\test_data\type\result


In [8]:
failures = []
start_all = time.time()

for idx, audio_path in enumerate(audio_files, start=1):
    print(f"[{idx}/{len(audio_files)}] Transcribing: {audio_path.name} ...", flush=True)
    t0 = time.time()
    try:
        result = model.transcribe(str(audio_path), language="ko", fp16=use_fp16)
        text = result.get("text", "").strip()
        out_txt = result_dir / f"{audio_path.stem}.txt"
        out_txt.write_text(text + "\n", encoding="utf-8")
        dt = time.time() - t0
        print(f" -> Saved: {out_txt.name} ({dt:.1f}s)")
    except Exception as exc:
        failures.append((audio_path.name, str(exc)))
        print(f" !! Failed: {audio_path.name}")
        traceback.print_exc()

elapsed = time.time() - start_all
print(f"\nDone. Elapsed: {elapsed:.1f}s")
if failures:
    print("Failures:")
    for name, err in failures:
        print(f" - {name}: {err}")
else:
    print("All files processed successfully.")

[1/5] Transcribing: A.flac ...


c:\Users\chan\anaconda3\envs\poc-ai\lib\site-packages\whisper\model.py:124: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  a = scaled_dot_product_attention(


 -> Saved: A.txt (46.2s)
[2/5] Transcribing: B.m4a ...
 -> Saved: B.txt (39.6s)
[3/5] Transcribing: C.mp3 ...
 -> Saved: C.txt (39.0s)
[4/5] Transcribing: D.ogg ...
 -> Saved: D.txt (44.3s)
[5/5] Transcribing: E.wav ...
 -> Saved: E.txt (38.7s)

Done. Elapsed: 207.7s
All files processed successfully.
